## Final dataset preparation
- Now, ones we have done cleaning and feature engineering, let's perform some final tasks :
1. Remove outliers
2. Encode categorical variables (if any)
3. standarize / normalize the data
4. Remove unnecessary columns for feeding into models

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [2]:
df = pd.read_csv('../data/03_engineered/satellites_engineered.csv')

In [3]:
df.head()

,OBJECT_NAME,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,...,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,ORBIT_PERIOD_SEC,SEMI_MAJOR_AXIS,ORBIT_HEIGHT,PERIGEE,APOGEE,ORBITAL_SPEED,AGE_SINCE_LAUNCH,SAT_TYPE
0,CALSPHERE 1,2025-12-04 15:36:43.212960,13.763517,0.002716,90.2215,67.2697,159.0699,309.0581,900,4466,...,1.211000e-05,0.0,6277.465223,7354.923145,983.923145,956.945702,996.900587,7.361725,61.925,C
1,CALSPHERE 2,2025-12-04 14:12:30.284352,13.528817,0.002040,90.2363,71.2062,74.4339,353.0789,902,82995,...,7.400000e-07,0.0,6386.367527,7439.741868,1068.741868,1046.564050,1076.919685,7.319640,61.925,E
2,TEMPSAT 1,2025-12-04 15:49:06.875904,13.335813,0.007139,89.9887,212.6762,75.2749,339.0456,1512,93412,...,8.900000e-07,0.0,6478.795009,7511.351540,1140.351540,1079.728002,1186.975079,7.284665,60.925,E
3,CALSPHERE 4A,2025-12-04 16:19:00.877728,13.362372,0.006822,89.9093,124.3111,292.9140,93.8021,1520,93673,...,1.750000e-06,0.0,6465.917684,7501.395143,1130.395143,1072.218375,1174.571911,7.289498,60.925,H
4,OPS 5712 (P/L 160),2025-12-04 16:45:34.701120,14.739996,0.000504,69.9162,153.7607,291.0755,68.9852,2826,3753,...,2.236400e-04,0.0,5861.602644,7026.399599,655.399599,644.855483,651.943715,7.531860,58.925,A


----
### 1. Removing outliers
##### Key question : Why even remove outliers if we are going to perform anomaly detection?


### 1️⃣ Garbage-in → Garbage-out

* ML models (Isolation Forest, KMeans, etc.) assume that most of your data is “normal” to learn patterns.
* If your dataset accidentally has **obvious errors** (e.g., height = 0 km, speed = 1e6 m/s, negative values), the model may learn wrong patterns or consider normal data as anomalous.

---

### 2️⃣ Helps define “normal”

* Anomaly detection models define anomalies relative to what is normal.
* If there are **obvious outliers**, your baseline “normal” distribution will be skewed, reducing detection accuracy for true anomalies like orbital maneuvers.

---

### 3️⃣ Prevents false positives

* Spotting obvious errors ensures you **don’t flag bad data as anomalies**.
* Real anomalies should reflect **interesting satellite behavior**, not just bad CSV entries or measurement glitches.

---

✅ **Summary:**

* Spotting anomalies early is about **data quality**, not about defeating your anomaly detection goal.
* Once the dataset is clean, your ML models can focus on **real, subtle anomalies** (like unusual delta in orbital height, speed, or maneuvers).




---

## ❗For prototyping purpose, we will not perform outlier removal step for now, later on we may perform that!

### 2. Encode categorical variables 

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12747 entries, 0 to 12746
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   OBJECT_NAME        12747 non-null  object 
 1   EPOCH              12747 non-null  object 
 2   MEAN_MOTION        12747 non-null  float64
 3   ECCENTRICITY       12747 non-null  float64
 4   INCLINATION        12747 non-null  float64
 5   RA_OF_ASC_NODE     12747 non-null  float64
 6   ARG_OF_PERICENTER  12747 non-null  float64
 7   MEAN_ANOMALY       12747 non-null  float64
 8   NORAD_CAT_ID       12747 non-null  int64  
 9   REV_AT_EPOCH       12747 non-null  int64  
 10  BSTAR              12747 non-null  float64
 11  MEAN_MOTION_DOT    12747 non-null  float64
 12  MEAN_MOTION_DDOT   12747 non-null  float64
 13  ORBIT_PERIOD_SEC   12747 non-null  float64
 14  SEMI_MAJOR_AXIS    12747 non-null  float64
 15  ORBIT_HEIGHT       12747 non-null  float64
 16  PERIGEE            127

- We only have one categorical feature, that's going to be feed into the models and that is 'satellite type'.
- So we will use OneHotEncoder to encode the feature

#### ❗Removing the columns not requiring scaling

In [5]:
df = df.drop(columns=['OBJECT_NAME', 'EPOCH', 'NORAD_CAT_ID'])

In [6]:
from sklearn.preprocessing import OneHotEncoder

In [7]:
# Fit and transform
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = encoder.fit_transform(df[['SAT_TYPE']])

# Get actual category names
encoded_cols = encoder.get_feature_names_out(['SAT_TYPE'])

# Convert to DataFrame with proper column names
df_encoded = pd.concat([df.drop('SAT_TYPE', axis=1), pd.DataFrame(X_encoded, columns=encoded_cols)], axis=1)


In [8]:
df_encoded.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,13.763517,0.002716,90.2215,67.2697,159.0699,309.0581,4466,0.001228,1.211000e-05,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13.528817,0.002040,90.2363,71.2062,74.4339,353.0789,82995,0.000097,7.400000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,13.335813,0.007139,89.9887,212.6762,75.2749,339.0456,93412,0.000160,8.900000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,13.362372,0.006822,89.9093,124.3111,292.9140,93.8021,93673,0.000319,1.750000e-06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,14.739996,0.000504,69.9162,153.7607,291.0755,68.9852,3753,0.003397,2.236400e-04,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# We combined Main dataframe (excluding the SAT_TYPE) + (X encoded (values) + encoded_cols feature names) 
X_encoded

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(12747, 24))

In [10]:
encoded_cols

array(['SAT_TYPE_A', 'SAT_TYPE_B', 'SAT_TYPE_C', 'SAT_TYPE_D',
       'SAT_TYPE_E', 'SAT_TYPE_F', 'SAT_TYPE_G', 'SAT_TYPE_H',
       'SAT_TYPE_J', 'SAT_TYPE_K', 'SAT_TYPE_L', 'SAT_TYPE_M',
       'SAT_TYPE_N', 'SAT_TYPE_P', 'SAT_TYPE_Q', 'SAT_TYPE_R',
       'SAT_TYPE_S', 'SAT_TYPE_T', 'SAT_TYPE_U', 'SAT_TYPE_V',
       'SAT_TYPE_W', 'SAT_TYPE_X', 'SAT_TYPE_Y', 'SAT_TYPE_Z'],
      dtype=object)

In [11]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12747 entries, 0 to 12746
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MEAN_MOTION        12747 non-null  float64
 1   ECCENTRICITY       12747 non-null  float64
 2   INCLINATION        12747 non-null  float64
 3   RA_OF_ASC_NODE     12747 non-null  float64
 4   ARG_OF_PERICENTER  12747 non-null  float64
 5   MEAN_ANOMALY       12747 non-null  float64
 6   REV_AT_EPOCH       12747 non-null  int64  
 7   BSTAR              12747 non-null  float64
 8   MEAN_MOTION_DOT    12747 non-null  float64
 9   MEAN_MOTION_DDOT   12747 non-null  float64
 10  ORBIT_PERIOD_SEC   12747 non-null  float64
 11  SEMI_MAJOR_AXIS    12747 non-null  float64
 12  ORBIT_HEIGHT       12747 non-null  float64
 13  PERIGEE            12747 non-null  float64
 14  APOGEE             12747 non-null  float64
 15  ORBITAL_SPEED      12747 non-null  float64
 16  AGE_SINCE_LAUNCH   127

---
### 3. Feature Scaling 

- **Numeric Features:**  
  Features like `SEMI_MAJOR_AXIS`, `ORBIT_HEIGHT`, `ORBITAL_SPEED`, etc., have different units and ranges. Scaling them using **StandardScaler** standardizes the values (mean=0, std=1) so that all numeric features contribute equally to distance-based algorithms like **KMeans** and **Isolation Forest**.

- **Categorical Features (One-Hot Encoded):**  
  Features like `SAT_TYPE` are converted into binary columns (0/1). These **do not require scaling**, as their values are already normalized and scaling would distort the categorical meaning.

- **Key Idea:**  
  - Scale numeric continuous features.  
  - Keep one-hot categorical features as-is.  
  This ensures the model correctly interprets distances and patterns without bias from different units.


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer # lets you apply different preprocessing to different columns

In [13]:
scaler = StandardScaler()

encoded_cols = [col for col in df_encoded.columns if col.startswith('SAT_TYPE_')]
numeric_cols = df_encoded.drop(columns = encoded_cols).columns

ct = ColumnTransformer([
    ('scaler', StandardScaler(), numeric_cols), # scale numeric feature
    ('pass', 'passthrough', encoded_cols) # keep encoded columns as is
])

df_scaled = ct.fit_transform(df_encoded)

In [14]:
df_scaled

array([[-2.02825677,  0.71237047,  1.36444466, ...,  0.        ,
         0.        ,  0.        ],
       [-2.40804928,  0.49161757,  1.36516035, ...,  0.        ,
         0.        ,  0.        ],
       [-2.72036981,  2.15645556,  1.35318707, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.77050048, -0.05708121, -0.50174289, ...,  0.        ,
         0.        ,  0.        ],
       [-0.05471021,  0.14385584,  1.72315465, ...,  0.        ,
         0.        ,  0.        ],
       [-1.01432985, 28.05343833, -0.24419164, ...,  0.        ,
         0.        ,  0.        ]], shape=(12747, 41))

In [15]:
# convert to dataframe                        # numeric_col is a series, so convert to a list 
df_scaled = pd.DataFrame(df_scaled, columns = numeric_cols.tolist() + encoded_cols)

In [16]:
df_scaled.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,-2.028257,0.712370,1.364445,-0.995301,-0.036004,1.179214,-0.671252,0.157480,-0.029430,-0.013178,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-2.408049,0.491618,1.365160,-0.958414,-0.932404,1.642944,5.245827,-0.035489,-0.033018,-0.013178,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-2.720370,2.156456,1.353187,0.367233,-0.923497,1.495113,6.030737,-0.024856,-0.032971,-0.013178,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-2.677392,2.053050,1.349347,-0.460794,1.381571,-1.088367,6.050403,0.002381,-0.032700,-0.013178,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.448113,-0.009803,0.382534,-0.184836,1.362099,-1.349797,-0.724976,0.527563,0.037320,-0.013178,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


- Save as model ready dataset

In [17]:
df_scaled.to_csv('../data/04_scaled/satellites_scaled.csv', index=False)

- Save the encoder also (for later use in isolation forest)

In [18]:
import joblib

joblib.dump(encoder, '../data/04_scaled/encoder_sat_type.joblib')

['../data/04_scaled/encoder_sat_type.joblib']

#### Now we are ready for Machine Learning models!